# Frame reconstruction: downstream test of simulated events
Pretrained **E2VID** and **FireNet** reconstruct intensity frames from the real events and from every method's events in `data_test_evt/<DATASET>/<SPLIT>/`. Nothing is trained. All the work is done by `benchmarks/recon_eval.py`; this notebook sets it up, runs it and shows the results. The dataset is a parameter, so any folder under `data_test_evt/` works.

**Protocol (same for every event source)**
- Frame *k* (k ≥ 1) is reconstructed from the events in [t<sub>k−1</sub>, t<sub>k</sub>), so it lines up with `frames/{k:06d}.png` and `frame_ts[k]` of the source sequence.
- Each window becomes a 5-bin voxel grid (bilinear in time, polarity ±1), with non-zero entries normalised to mean 0 and std 1, as in E2VID.
- All sources of a sequence run as one batch with one recurrent state each, so every method gets identical windows, padding and state handling. The state is reset per sequence and frames k < `WARMUP` are not scored.
- **V2V is left out**: it stores one summed voxel per frame interval, without the timing inside the interval that the networks need.

**Metrics** (images min–max normalised before scoring)
- `agreement_*`: reconstruction from a method's events vs reconstruction from the **real** events, same frame and network. **Main result**, independent of the GT frames.
- `gt_*`: reconstruction vs the ground-truth frame. Secondary, because the simulators were given these frames. The `real` row is the reference level.
- MSE ↓, SSIM ↑ (Gaussian 11×11, σ 1.5), LPIPS ↓ (AlexNet).

**Outputs**
- Sample frames: `data_test_recon/<DATASET>/<SPLIT>/<network>/<method>/<sequence>/{k:06d}.png`, plus `recon.json`. The method `real` is the reconstruction from real events.
- Reports: `reports/recon/<DATASET>/<SPLIT>/<timestamp>/` with `per_frame.csv`, `per_sequence.csv`, `summary.csv`, `sanity.csv`, `wilcoxon.csv`, `config.json`, and the qualitative figures.

**Setup (once):** `pip install lpips`, plus E2VID and FireNet checkpoints and the model code that defines them (EVREAL, or rpg_e2vid and its FireNet branch). Point to them in the next cell.

In [ ]:
from pathlib import Path
import os, sys

REPO = Path.cwd().resolve()
while not (REPO / "benchmarks" / "recon_eval.py").is_file():
    REPO = REPO.parent
sys.path.insert(0, str(REPO / "benchmarks"))
import recon_eval

# ---- settings -------------------------------------------------------------
DATASET = "EDS"            # any folder under data_test_evt/: "EDS", "HQ-EVFI", "BS-ERGB", ...
SPLIT = "test"
METHODS = None             # None = every method found for the dataset (V2V is always left out)
SEQUENCES = None           # None = all sequences
NETWORKS = ["e2vid", "firenet"]
WARMUP = 10
SAVE = "samples"           # "none" | "samples" | "all"
SAMPLES = 5                # frames saved per sequence with SAVE = "samples"
DEVICES = ["cuda:0", "cuda:0"]  # one worker per entry; two workers on one A100 overlap loading and compute
RUN_DIR = None             # set to an existing report folder to resume it or only view its results

os.environ.setdefault("RECON_E2VID_CKPT", "/path/to/E2VID_lightweight.pth.tar")
os.environ.setdefault("RECON_FIRENET_CKPT", "/path/to/firenet_1000.pth.tar")
os.environ.setdefault("EVREAL_ROOT", "/path/to/EVREAL")   # code root with model/model.py
# os.environ["RECON_E2VID_CODE"] = "..."; os.environ["RECON_FIRENET_CODE"] = "..."  # if they differ

argv = [DATASET, "--split", SPLIT, "--networks", *NETWORKS, "--warmup", str(WARMUP),
        "--save", SAVE, "--samples", str(SAMPLES), "--devices", *DEVICES]
if METHODS: argv += ["--methods", *METHODS]
if SEQUENCES: argv += ["--sequences", *SEQUENCES]
if RUN_DIR: argv += ["--run-dir", str(RUN_DIR)]
print("python benchmarks/recon_eval.py " + " ".join(argv))

## Run
Long runs are safer in a terminal (`tmux`) with the command printed above. Use the same `--run-dir` to resume: finished sequences are skipped. To only view an existing run, set `RUN_DIR` above and skip this cell.

In [ ]:
run_dir = recon_eval.main(argv)

In [ ]:
import json
import pandas as pd
from IPython.display import display, Image as IPImage

run_dir = Path(RUN_DIR) if RUN_DIR else run_dir
recon_eval.summarize(run_dir)
summary = pd.read_csv(run_dir / "summary.csv")
config = json.loads((run_dir / "config.json").read_text())
print(run_dir)
print("methods:", config["methods"])
print("excluded:", config["excluded"])

def table(kind):
    cols = {}
    for m in recon_eval.METRICS:
        c = f"{kind}_{m}"
        arrow = "↑" if recon_eval.HIGHER_IS_BETTER[m] else "↓"
        cols[f"{m.upper()} {arrow}"] = summary.apply(
            lambda r: f"{r[c+'_mean']:.4f} ± {r[c+'_std']:.4f} ({int(r[c+'_rank'])})"
            if pd.notna(r[c + "_mean"]) else "", axis=1)
    out = pd.concat([summary[["network", "method", "sequences", "frames"]], pd.DataFrame(cols)], axis=1)
    if kind == "agreement":
        out = out[out.method != recon_eval.REAL]
    return out.set_index(["network", "method"])

## Main result: agreement with the reconstruction from real events
Mean ± std over sequences (each sequence weighted equally), rank in brackets.

In [ ]:
display(table("agreement"))

## Secondary: against the ground-truth frames
The simulators saw these frames, so read this next to the `real` row.

In [ ]:
display(table("gt"))

## Sanity check of the inputs
Event count, rate, ON fraction and empty windows per source and sequence. `status = missing` means a method has no output for that sequence.

In [ ]:
sanity = pd.read_csv(run_dir / "sanity.csv")
display(sanity.groupby("method", sort=False)[["events", "rate_hz", "on_fraction", "empty_windows"]].mean())
display(sanity[sanity.status != "ok"])

## Best Evtween variant vs best baseline
Paired Wilcoxon signed-rank test over sequences, per network and metric. With fewer than 6 sequences the two-sided p-value cannot go below 0.05.

In [ ]:
display(pd.read_csv(run_dir / "wilcoxon.csv"))

## Qualitative comparison
The same frame reconstructed from every source, next to the ground truth. It uses the saved sample frames.

In [ ]:
sequence = config["sequences"][0]
for network in config["networks"]:
    path = recon_eval.qualitative(run_dir, sequence, network=network)
    display(IPImage(filename=str(path)))